# Table of Contents
- [Problem Definition](#intro)
- [Modules/Libraries](#modules)
- [Exploratory Data Analysis](#eda)
- [Data Cleanup](#cleanup)
- [NLP](#nlp)
- [Conclusion](#conclusion)

## 👨🏻‍💻 Problem Definition <a id='intro'></a>

- Describe problem

## ⚒️ Import data & modules <a id='modules'></a>
We will use some standard modules for data analysis, viz libraries, NLP tools.

In [14]:
# Standard Tools
import pandas as pd
import matplotlib.pyplot as plt

# NLP Tools
import re
import nltk
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

In [15]:
# Necessary Resources from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ipozdnyakov/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ipozdnyakov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/ipozdnyakov/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## 🔮 Exploratory Data Analysis (EDA) <a id='eda'></a>
Let's take a first look at the data and try to infer some relationship between variables and decide if we need some additional cleanup.

The Amazon reviews polarity dataset is constructed by taking review score 1 and 2 as negative, and 4 and 5 as positive. Samples of score 3 is ignored. In the dataset, class 1 is the negative and class 2 is the positive. Each class has 1,800,000 training samples and 200,000 testing samples.

The files train.csv and test.csv contain all the training samples as comma-sparated values. There are 3 columns in them, corresponding to class index (1 or 2), review title and review text. The review title and text are escaped using double quotes ("), and any internal double quote is escaped by 2 double quotes (""). New lines are escaped by a backslash followed with an "n" character, that is "\n".

In [17]:
# Use Kaggle API to import the dataset
#path = kagglehub.dataset_download('kritanjalijain/amazon-reviews')
#print("Path to dataset files:", path)
path = '/Users/ipozdnyakov/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2'
column_names = ['class_index', 'review_title', 'review_text']
train_df = pd.read_csv(path + '/train.csv', header=None, names=column_names)
test_df = pd.read_csv(path + '/test.csv', header=None, names=column_names)

# Combine Datasets for Processing
train_df['source'] = 'train'
test_df['source'] = 'test'
df = pd.concat([train_df, test_df], ignore_index=True)

In [18]:
# Check for NAs in columns
print(df.isnull().sum())

Checking for missing values in each column:
class_index       0
review_title    231
review_text       0
source            0
dtype: int64


In [19]:
# Check the structure of the Dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000000 entries, 0 to 3999999
Data columns (total 4 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   class_index   int64 
 1   review_title  object
 2   review_text   object
 3   source        object
dtypes: int64(1), object(3)
memory usage: 122.1+ MB


In [10]:
df.head()

array([2, 1])

## 🧹 Data Cleanup <a id='cleanup'></a>
For every NLP project, it is imporant to make sure that the text data is as clean as possible. We will need to get rid of special characters, extract word stems, tokenize words. The cleaner the data, the better the results!